In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
from pyspark.sql import SparkSession

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté
# On fusionne les deux types de données du Bronze (BATCH et STREAMING)

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA bronze")

bronze_landing = "/Volumes/main/bronze/bronze_volume/landing_zone"
bronze_checkpoint = (
    "/Volumes/main/bronze/bronze_volume"
    "/_checkpoints/bronze_stream"
)

# Schéma Kaggle-like (identique à ton générateur)
schema = StructType([
    StructField("Time", DoubleType(), True),
    StructField("V1", DoubleType(), True),
    StructField("V2", DoubleType(), True),
    StructField("V3", DoubleType(), True),
    StructField("V4", DoubleType(), True),
    StructField("V5", DoubleType(), True),
    StructField("V6", DoubleType(), True),
    StructField("V7", DoubleType(), True),
    StructField("V8", DoubleType(), True),
    StructField("V9", DoubleType(), True),
    StructField("V10", DoubleType(), True),
    StructField("V11", DoubleType(), True),
    StructField("V12", DoubleType(), True),
    StructField("V13", DoubleType(), True),
    StructField("V14", DoubleType(), True),
    StructField("V15", DoubleType(), True),
    StructField("V16", DoubleType(), True),
    StructField("V17", DoubleType(), True),
    StructField("V18", DoubleType(), True),
    StructField("V19", DoubleType(), True),
    StructField("V20", DoubleType(), True),
    StructField("V21", DoubleType(), True),
    StructField("V22", DoubleType(), True),
    StructField("V23", DoubleType(), True),
    StructField("V24", DoubleType(), True),
    StructField("V25", DoubleType(), True),
    StructField("V26", DoubleType(), True),
    StructField("V27", DoubleType(), True),
    StructField("V28", DoubleType(), True),
    StructField("Amount", DoubleType(), True),
    StructField("Class", IntegerType(), True)
])

# Lecture streaming des CSV Kaggle-like
stream_df = (
    spark.readStream
         .schema(schema)
         .option("header", False)
         .csv(bronze_landing)
)

stream_df.printSchema()  # pour vérifier que tout matche bien

# Écriture vers la table Delta Bronze
query = (
    stream_df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", bronze_checkpoint)
             .trigger(once=True)  # ou availableNow=True selon ton cluster
             .table("transactions_bronze_stream")
)